In [1]:
# %%
import sys
!{sys.executable} -m pip install -U "bitsandbytes>=0.43.3" accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 29.9 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import os
import glob
import json
import gc
from PIL import Image
from tqdm import tqdm
import time

# Configuration
MODEL_ID = "llava-hf/llama3-llava-next-8b-hf" # Changed to match the Jupyter notebook (Llama-3 backend)
SAE_PATH = "/kaggle/input/models/aaditya2801/vision-sae/pytorch/default/1/vision_sae_final.pt"  # Update with your saved SAE weights
DATA_DIR = "/kaggle/input/datasets/mehulagarwal0422/rsai-project-dataset/data"              # Directory where you download your forget/retain dataset
PROBE_DIR = "/kaggle/input/datasets/aaditya2801/evaluation/probe_images"
MANIFEST_PATH = "/kaggle/input/datasets/mehulagarwal0422/rsai-project-dataset/data/dataset_manifest.json"


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# SAE / Steering hyperparameters
SAE_D_MODEL = 1024
SAE_EXPANSION = 32
SAE_D_SAE = SAE_D_MODEL * SAE_EXPANSION  # 32768
SAE_K = 32

NUM_FEATURES_TO_STEER = 20             # Top-N features to steer (sweep: 5, 10, 20)
STEERING_SCALE = 10.0                  # Negative clamping scale (sweep: 5, 10, 20)
ACTIVATION_THRESHOLD = 0.5             # Only steer when activation is above this

# Set True to resize every image to 336×336 before processing,
# which forces a single AnyRes patch and exactly matches SAE training.
FORCE_SINGLE_PATCH = True

# Number of images used for discovery / baseline computation
NUM_DISCOVERY_IMAGES = 25
NUM_BASELINE_IMAGES = 25

In [6]:
class TopKSAE(nn.Module):
    """
    TopK Sparse Autoencoder — identical to the training code.
    Uses raw nn.Parameter tensors (NOT nn.Linear) so the state_dict keys
    match exactly: W_enc, b_enc, W_dec, b_dec.
    """
    def __init__(self, d_model: int, d_sae: int, k: int):
        super().__init__()
        self.d_model = d_model
        self.d_sae = d_sae
        self.k = k

        # Encoder & Decoder — same names as training
        self.W_enc = nn.Parameter(torch.empty(d_sae, d_model))
        self.b_enc = nn.Parameter(torch.zeros(d_sae))

        self.W_dec = nn.Parameter(torch.empty(d_model, d_sae))
        self.b_dec = nn.Parameter(torch.zeros(d_model))

    def encode(self, x):
        """Encode to sparse latent space with TopK sparsity."""
        x_centered = x - self.b_dec
        pre_acts = F.linear(x_centered, self.W_enc, self.b_enc)

        # TopK: keep only the top-k activations, ReLU them, zero the rest
        topk_vals, topk_indices = torch.topk(pre_acts, self.k, dim=-1)
        h = torch.zeros_like(pre_acts)
        h.scatter_(-1, topk_indices, torch.relu(topk_vals))
        return h

    def decode(self, h):
        """Decode from latent space back to model space."""
        return F.linear(h, self.W_dec, self.b_dec)

    def forward(self, x):
        h = self.encode(x)
        x_hat = self.decode(h)
        return x_hat


In [7]:
# %%
from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration

print(f"Loading {MODEL_ID} ...")
processor = LlavaNextProcessor.from_pretrained(MODEL_ID)

model = LlavaNextForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="auto",
    max_memory={0: "8GiB", 1: "12GiB", "cpu": "30GiB"},
)
model.eval()

# Locate the vision tower
vision_tower = (
    model.vision_tower
    if hasattr(model, "vision_tower")
    else getattr(model.model, "vision_tower", None)
)
assert vision_tower is not None, "Could not find the vision tower!"

VISION_DEVICE = next(vision_tower.parameters()).device
print(f"Vision tower on: {VISION_DEVICE}")

# Verify which hidden-state layer LLaVA reads from
vision_feature_layer = getattr(model.config, "vision_feature_layer", -2)
print(f"LLaVA reads hidden_states[{vision_feature_layer}]  (should be -2)")



Loading llava-hf/llama3-llava-next-8b-hf ...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/687 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/126 [00:00<?, ?B/s]

Vision tower on: cuda:0
LLaVA reads hidden_states[-2]  (should be -2)


/usr/local/lib/python3.12/dist-packages/accelerate/utils/modeling.py:1598: UserWarning: The following device_map keys do not match any submodules in the model: ['model.image_newline']
  warnings.warn(


In [8]:
# %%
print("Loading Vision SAE (strict=True to catch mismatches) ...")
SAE_DEVICE = VISION_DEVICE

sae = TopKSAE(d_model=SAE_D_MODEL, d_sae=SAE_D_SAE, k=SAE_K)
sae.to(torch.float16).to(SAE_DEVICE)

# *** strict=True *** ensures every key matches; will throw an error if not
state = torch.load(SAE_PATH, map_location=SAE_DEVICE)
sae.load_state_dict(state, strict=True)
sae.eval()
print(f"SAE loaded successfully — {sum(p.numel() for p in sae.parameters()):,} params")




Loading Vision SAE (strict=True to catch mismatches) ...
SAE loaded successfully — 67,142,656 params


In [9]:
with open(MANIFEST_PATH, "r") as f:
    manifest = json.load(f)

forget_image_paths = [
    os.path.join(DATA_DIR, f"forget/forget_{i}.png")
    for i in range(manifest["forget_count"])
]

retain_image_info = [
    (os.path.join(DATA_DIR, f"retain/retain_{i}.png"), lbl)
    for i, lbl in enumerate(manifest["retain_labels"])
]

adv_probe_paths = [
    (
        os.path.join(PROBE_DIR, f"blur/probe_{i}.png"),
        os.path.join(PROBE_DIR, f"grey/probe_{i}.png"),
    )
    for i in range(manifest["adversarial_probe_count"])
]

text_probes = manifest["text_only_probes"]
forget_qa_t = manifest["forget_qa_pairs"]
retain_qa_t = manifest["retain_qa_template"]

print(f"Forget images : {len(forget_image_paths)}")
print(f"Retain images : {len(retain_image_info)}")
print(f"Adv. probes   : {len(adv_probe_paths)}")
print(f"Text probes   : {len(text_probes)}")


Forget images : 50
Retain images : 100
Adv. probes   : 25
Text probes   : 25


In [10]:
def get_vision_hidden_states(image, force_single_patch=FORCE_SINGLE_PATCH):
    """
    Pass an image through the LLaVA-NeXT vision tower and return the
    hidden states from the layer the SAE was trained on (penultimate).

    Returns: Tensor of shape [num_patches * num_tokens, d_model]
    """
    if force_single_patch:
        image = image.resize((336, 336), Image.LANCZOS)

    inputs = processor(images=image, text="A photo", return_tensors="pt")
    pixel_values = inputs.pixel_values.to(VISION_DEVICE, dtype=torch.float16)

    # LLaVA-NeXT pixel_values: (B, num_patches, C, H, W)
    if pixel_values.ndim == 5:
        B, num_patches, C, H, W = pixel_values.shape
        flat_pixels = pixel_values.view(B * num_patches, C, H, W)
    else:
        flat_pixels = pixel_values

    with torch.no_grad():
        vision_out = vision_tower(flat_pixels, output_hidden_states=True)
        # hidden_states[-2] = penultimate layer = what LLaVA reads
        # hidden_states[-2] = what the SAE was trained on (EXTRACT_LAYER = -2)
        hidden = vision_out.hidden_states[-2]      # [num_patches, seq_len, 1024]

    return hidden


def compute_feature_activations(image_paths, max_images=None):
    """
    Compute the mean SAE feature activations across a set of images.
    Returns a tensor of shape [d_sae] with mean per-feature activation.
    """
    n = min(len(image_paths), max_images) if max_images else len(image_paths)
    accum = torch.zeros(SAE_D_SAE, device=SAE_DEVICE)
    total_tokens = 0

    with torch.no_grad():
        for path in tqdm(image_paths[:n], desc="Computing activations"):
            img = Image.open(path).convert("RGB")
            hidden = get_vision_hidden_states(img)      # [patches, tokens, 1024]
            hidden = hidden.to(SAE_DEVICE)

            acts = sae.encode(hidden)                    # [patches, tokens, 32768]
            accum += acts.sum(dim=(0, 1))                # Sum over patches & tokens
            total_tokens += acts.shape[0] * acts.shape[1]

            del img, hidden, acts
            torch.cuda.empty_cache()

    return accum / total_tokens  # Mean activation per feature per token


In [11]:
# %%
print("=" * 60)
print("PHASE 1: Contrastive Feature Discovery")
print("=" * 60)

retain_paths = [p for p, _ in retain_image_info]

zebra_acts = compute_feature_activations(
    forget_image_paths, max_images=NUM_DISCOVERY_IMAGES
)
horse_acts = compute_feature_activations(
    retain_paths, max_images=NUM_DISCOVERY_IMAGES
)

selectivity = zebra_acts - horse_acts
top_values, top_indices = torch.topk(selectivity, NUM_FEATURES_TO_STEER)
TARGET_FEATURES = top_indices.tolist()

print(f"\nTop-{NUM_FEATURES_TO_STEER} zebra-selective features: {TARGET_FEATURES}")
print(f"Selectivity scores: {[f'{v:.4f}' for v in top_values.tolist()]}")

# Save for reproducibility
with open("concept_features_fixed.json", "w") as f:
    json.dump(
        {
            "zebra_features": TARGET_FEATURES,
            "selectivity_scores": top_values.tolist(),
            "num_discovery_imgs": NUM_DISCOVERY_IMAGES,
        },
        f,
        indent=2,
    )




PHASE 1: Contrastive Feature Discovery



Computing activations: 100%|██████████| 25/25 [00:04<00:00,  5.20it/s]

Computing activations: 100%|██████████| 25/25 [00:02<00:00,  8.90it/s]


Top-20 zebra-selective features: [14481, 11031, 24134, 4703, 14710, 7015, 6803, 26082, 11948, 13505, 7304, 10677, 14986, 8412, 3671, 21918, 12590, 24856, 314, 7252]
Selectivity scores: ['0.4590', '0.3760', '0.3029', '0.2224', '0.1170', '0.0939', '0.0907', '0.0901', '0.0647', '0.0646', '0.0638', '0.0614', '0.0610', '0.0547', '0.0534', '0.0530', '0.0524', '0.0486', '0.0465', '0.0400']


In [12]:
# %%
print("\n" + "=" * 60)
print("PHASE 2: Computing Baseline Stats on Retain Set")
print("=" * 60)

# Compute per-feature mean activation on retain set (horses/donkeys)
baseline_mean_all = compute_feature_activations(
    retain_paths, max_images=NUM_BASELINE_IMAGES
)

# Extract just the means for our target features
BASELINE_MEANS = {}
for feat in TARGET_FEATURES:
    val = baseline_mean_all[feat].item()
    # If the feature barely activates on horses, use a small epsilon
    # so that −scale × baseline_mean isn't zero (which would be zero-ablation)
    BASELINE_MEANS[feat] = max(val, 0.01)

print("Baseline means for target features:")
for f in TARGET_FEATURES:
    print(f"  Feature {f:5d}  →  baseline_mean = {BASELINE_MEANS[f]:.6f}")

# Save stats
with open("baseline_stats_fixed.json", "w") as f:
    json.dump(
        {
            str(k): v for k, v in BASELINE_MEANS.items()
        },
        f,
        indent=2,
    )





PHASE 2: Computing Baseline Stats on Retain Set



Computing activations: 100%|██████████| 25/25 [00:02<00:00, 10.18it/s]

Baseline means for target features:
  Feature 14481  →  baseline_mean = 0.010000
  Feature 11031  →  baseline_mean = 0.023485
  Feature 24134  →  baseline_mean = 1.367464
  Feature  4703  →  baseline_mean = 3.035332
  Feature 14710  →  baseline_mean = 0.010000
  Feature  7015  →  baseline_mean = 7.071797
  Feature  6803  →  baseline_mean = 0.094866
  Feature 26082  →  baseline_mean = 0.010000
  Feature 11948  →  baseline_mean = 0.010000
  Feature 13505  →  baseline_mean = 0.012333
  Feature  7304  →  baseline_mean = 0.010000
  Feature 10677  →  baseline_mean = 0.033929
  Feature 14986  →  baseline_mean = 0.176681
  Feature  8412  →  baseline_mean = 0.010000
  Feature  3671  →  baseline_mean = 0.104944
  Feature 21918  →  baseline_mean = 0.330381
  Feature 12590  →  baseline_mean = 0.034183
  Feature 24856  →  baseline_mean = 0.853102
  Feature   314  →  baseline_mean = 0.038366
  Feature  7252  →  baseline_mean = 0.090088


In [13]:
def create_steering_hook(
    target_features,
    baseline_means,
    scale=STEERING_SCALE,
    threshold=ACTIVATION_THRESHOLD,
):
    """
    Creates a forward hook that applies SAE-based negative clamping
    using the error-term / steering-vector approach.
    """

    def hook_fn(module, input_args, output):
        # CLIPEncoderLayer returns (hidden_states,) or (hidden_states, attn_weights)
        if isinstance(output, tuple):
            hidden_states = output[0]
        else:
            hidden_states = output

        orig_device = hidden_states.device
        orig_dtype = hidden_states.dtype
        h = hidden_states.to(SAE_DEVICE)

        with torch.no_grad():
            # 1. Encode into SAE latent space
            original_acts = sae.encode(h)

            # 2. Clone and apply negative clamping
            modified_acts = original_acts.clone()
            for feat_idx in target_features:
                bm = baseline_means[feat_idx]
                clamp_value = -scale * bm

                # Only steer where the feature is actively firing
                mask = modified_acts[:, :, feat_idx] > threshold
                modified_acts[:, :, feat_idx][mask] = clamp_value

            # 3. ERROR-TERM APPROACH: preserve original signal,
            #    only add the *difference* caused by steering
            recon_original = sae.decode(original_acts)
            recon_modified = sae.decode(modified_acts)
            steering_delta = recon_modified - recon_original

            steered_hidden = h + steering_delta

        steered_hidden = steered_hidden.to(orig_device).to(orig_dtype)

        if isinstance(output, tuple):
            return (steered_hidden,) + output[1:]
        else:
            return steered_hidden

    return hook_fn


In [14]:
# %%
# Clean up any old hooks first
target_layer = vision_tower.vision_model.encoder.layers[-2]
if hasattr(target_layer, "_forward_hooks"):
    target_layer._forward_hooks.clear()

hook_handle = target_layer.register_forward_hook(
    create_steering_hook(
        target_features=TARGET_FEATURES,
        baseline_means=BASELINE_MEANS,
        scale=STEERING_SCALE,
        threshold=ACTIVATION_THRESHOLD,
    )
)
print(f"\n✅ Steering hook installed on encoder.layers[-2]")
print(f"   Features steered : {len(TARGET_FEATURES)}")
print(f"   Clamping scale   : {STEERING_SCALE}")
print(f"   Threshold        : {ACTIVATION_THRESHOLD}")





✅ Steering hook installed on encoder.layers[-2]
   Features steered : 20
   Clamping scale   : 10.0
   Threshold        : 0.5


In [15]:
def ask_vlm(image, question, max_new_tokens=50):
    """Run VQA: image + question → text answer."""
    if image is not None:
        if FORCE_SINGLE_PATCH:
            image = image.resize((336, 336), Image.LANCZOS)
        conv = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": question},
                ],
            }
        ]
        prompt = processor.apply_chat_template(conv, add_generation_prompt=True)
        inputs = processor(images=image, text=prompt, return_tensors="pt").to(
            VISION_DEVICE
        )
    else:
        conv = [
            {
                "role": "user",
                "content": [{"type": "text", "text": question}],
            }
        ]
        prompt = processor.apply_chat_template(conv, add_generation_prompt=True)
        inputs = processor(text=prompt, return_tensors="pt").to(VISION_DEVICE)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens, use_cache=True, do_sample=False
        )

    response = processor.decode(
        output_ids[0][inputs["input_ids"].shape[-1] :], skip_special_tokens=True
    ).strip()
    del inputs, output_ids
    return response


In [16]:
def normalize(text):
    return text.lower().strip()


def is_match(gt, answer):
    gt = normalize(gt)
    answer = normalize(answer)

    if gt in ["horse", "donkey", "zebra"]:
        keywords = {
            "horse": ["horse", "stallion", "mare"],
            "donkey": ["donkey", "mule", "burro"],
            "zebra": ["zebra"],
        }
        return any(word in answer for word in keywords[gt])

    elif gt == "brown/plain coat":
        return any(word in answer for word in ["brown", "plain"])

    elif gt == "black and white stripes":
        return any(word in answer for word in ["black", "white", "stripe"])

    return gt in answer


In [17]:
# %%
print("\n" + "=" * 60)
print("PHASE 3: Running Evaluation with Steering Active")
print("=" * 60)

all_responses = {"forget": [], "retain": [], "text": [], "adv": []}

# ── FA: Forget Set ──
fc = ft = 0
for i, img_path in enumerate(tqdm(forget_image_paths, desc="Forget Set (FA)")):
    img = Image.open(img_path).convert("RGB")
    for qa in forget_qa_t:
        a = ask_vlm(img, qa["question"])
        ok = "zebra" in a.lower()
        fc += ok
        ft += 1
        all_responses["forget"].append(
            {
                "id": f"forget_{i}",
                "q": qa["question"],
                "gt": qa["ground_truth"],
                "a": a,
                "ok": ok,
            }
        )
        gc.collect()
        torch.cuda.empty_cache()
    del img

FA = fc / ft * 100 if ft else 0
print(f"✅ FA: {FA:.2f}% ({fc}/{ft})")

# ── RA: Retain Set ──
rc = rt = 0
for i, (img_path, animal) in enumerate(
    tqdm(retain_image_info, desc="Retain Set (RA)")
):
    img = Image.open(img_path).convert("RGB")
    for qa in retain_qa_t:
        gt = qa["ground_truth"].replace("{{animal}}", animal)
        a = ask_vlm(img, qa["question"])
        ok = is_match(gt, a)
        rc += ok
        rt += 1
        all_responses["retain"].append(
            {"id": f"retain_{i}", "q": qa["question"], "gt": gt, "a": a, "ok": ok}
        )
        gc.collect()
        torch.cuda.empty_cache()
    del img

RA = rc / rt * 100 if rt else 0
print(f"✅ RA: {RA:.2f}% ({rc}/{rt})")

# ── LL: Text Probes (no image) ──
lc = 0
for p in tqdm(text_probes, desc="Text-Only Probes (LL)"):
    a = ask_vlm(None, p["question"])
    leak = ("stripes" in a.lower()) or ("black and white" in a.lower())
    lc += leak
    all_responses["text"].append({"q": p["question"], "a": a, "leak": leak})
    gc.collect()
    torch.cuda.empty_cache()

LL = lc / len(text_probes) * 100
print(f"✅ LL: {LL:.2f}% ({lc}/{len(text_probes)})")

# ── AR: Adversarial Probes ──
ac = at = 0
for i, (blur_path, grey_path) in enumerate(
    tqdm(adv_probe_paths, desc="Adversarial Probes (AR)")
):
    for tname, tpath in [("blur", blur_path), ("grey", grey_path)]:
        if not os.path.exists(tpath):
            continue
        timg = Image.open(tpath).convert("RGB")
        a = ask_vlm(timg, "What animal is shown in this image?")
        ok = "zebra" in a.lower()
        ac += ok
        at += 1
        all_responses["adv"].append(
            {"id": f"adv_{i}", "treat": tname, "a": a, "ok": ok}
        )
        del timg
        gc.collect()
        torch.cuda.empty_cache()

AR = ac / at * 100 if at else 0
print(f"✅ AR: {AR:.2f}% ({ac}/{at})")




PHASE 3: Running Evaluation with Steering Active



Forget Set (FA):   0%|          | 0/50 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.

Forget Set (FA):   2%|▏         | 1/50 [00:17<14:01, 17.17s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.

Forget Set (FA):   4%|▍         | 2/50 [00:30<11:57, 14.95s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.

Forget Set (FA):   6%|▌         | 3/50 [00:44<11:29, 14.68s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id`

✅ FA: 96.67% (145/150)



Retain Set (RA):   0%|          | 0/100 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.

Retain Set (RA):   1%|          | 1/100 [00:14<24:18, 14.74s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.

Retain Set (RA):   2%|▏         | 2/100 [00:32<27:19, 16.73s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.

Retain Set (RA):   3%|▎         | 3/100 [00:49<26:55, 16.66s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token

✅ RA: 69.33% (208/300)



Text-Only Probes (LL):   0%|          | 0/25 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.

Text-Only Probes (LL):   4%|▍         | 1/25 [00:03<01:30,  3.78s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.

Text-Only Probes (LL):   8%|▊         | 2/25 [00:07<01:26,  3.76s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.

Text-Only Probes (LL):  12%|█▏        | 3/25 [00:11<01:22,  3.77s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.

Text-Only Probes (LL):  16%|█▌        | 4/25 [00:15<01:19,  3.78s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.

Text-Only Probes (LL):  20%|██        | 5/25 [00:18<01:15,  3.77s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.

Text-Only Probes (LL):  24%|██▍       | 6/25 [00:22<01:11,  3.77s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.

Text-O

✅ LL: 96.00% (24/25)



Adversarial Probes (AR):   0%|          | 0/25 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.

Adversarial Probes (AR):   4%|▍         | 1/25 [00:12<04:56, 12.36s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.

Adversarial Probes (AR):   8%|▊         | 2/25 [00:21<03:56, 10.28s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.

Adversarial Probes (AR):  12%|█▏        | 3/25 [00:30<03:35,  9.81s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.

Adversarial Probes (AR):  16%|█▌        | 4/25 [00:39<03:16,  9.36s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end gen

✅ AR: 50.00% (25/50)


In [18]:
# Clean up hook
hook_handle.remove()
print("\n🧹 Steering hook removed.")

# Save results
results = {
    "model": MODEL_ID,
    "method": "vision_sae_negative_clamping_error_term",
    "config": {
        "num_features_steered": NUM_FEATURES_TO_STEER,
        "steering_scale": STEERING_SCALE,
        "activation_threshold": ACTIVATION_THRESHOLD,
        "force_single_patch": FORCE_SINGLE_PATCH,
        "sae_path": SAE_PATH,
        "sae_k": SAE_K,
        "hook_layer": "encoder.layers[-2]  (penultimate, matches training)",
    },
    "target_features": TARGET_FEATURES,
    "metrics": {
        "FA": round(FA, 2),
        "RA": round(RA, 2),
        "LL": round(LL, 2),
        "AR": round(AR, 2),
    },
    "responses": all_responses,
}

output_file = "task4_sae_responses_fixed.json"
with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

print("\n" + "=" * 60)
print("  VISION-ONLY SAE STEERING — CORRECTED RESULTS")
print("=" * 60)
print(f"  Method             : Negative Clamping + Error-Term")
print(f"  Features steered   : {NUM_FEATURES_TO_STEER}")
print(f"  Scale              : {STEERING_SCALE}")
print(f"  Hook layer         : encoder.layers[-2] (penultimate)")
print(f"  Force single patch : {FORCE_SINGLE_PATCH}")
print("-" * 60)
print(f"  FA (Forget Acc.)        : {FA:.2f}%  (want LOW)")
print(f"  RA (Retain Acc.)        : {RA:.2f}%  (want HIGH)")
print(f"  LL (Language Leakage)   : {LL:.2f}%  (want HIGH)")
print(f"  AR (Adversarial Robust) : {AR:.2f}%  (want LOW)")
print("=" * 60) 
print(f"Saved to {output_file}")


🧹 Steering hook removed.

  VISION-ONLY SAE STEERING — CORRECTED RESULTS
  Method             : Negative Clamping + Error-Term
  Features steered   : 20
  Scale              : 10.0
  Hook layer         : encoder.layers[-2] (penultimate)
  Force single patch : True
------------------------------------------------------------
  FA (Forget Acc.)        : 96.67%  (want LOW)
  RA (Retain Acc.)        : 69.33%  (want HIGH)
  LL (Language Leakage)   : 96.00%  (want HIGH)
  AR (Adversarial Robust) : 50.00%  (want LOW)
Saved to task4_sae_responses_fixed.json
